# Sailor-7B constrained-track pipeline

Run All or Save & Run All. Enable Internet, select GPU T4 ×2, attach the private canonical dataset and inference input, and grant access to `GITHUB_TOKEN` in Kaggle Secrets. Optional `WANDB_API_KEY` enables online tracking; without it, tracking stays offline.

The notebook runs `configs/experiment.json` from the latest committed `kaggle-linux` branch. Change experiments through chat and commits to that recipe or the two training configs. There are no notebook experiment switches. Each run saves the exact Git commit, resolved configs, dataset hashes, and a ready-to-submit `results.zip`. Keep corpus-bearing outputs private.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
PROJECT_SUBDIR = "."
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Git runs as a subprocess; notebook execution needs no kernel restart.
# Always clone into a fresh directory; old source edits and outputs are untouched.
from urllib.parse import urlsplit
from contextlib import contextmanager
import tempfile
import os

@contextmanager
def github_git():
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    token = None
    with tempfile.TemporaryDirectory() as temporary:
        if urlsplit(REPO_URL).hostname == "github.com":
            from kaggle_secrets import UserSecretsClient
            try:
                token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception:
                raise RuntimeError("Add GITHUB_TOKEN in Kaggle Secrets and enable notebook access. "
                                   "The private repository requires Contents: Read-only permission.") from None
            if not token:
                raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
            askpass = Path(temporary) / "git-askpass.sh"
            askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                               '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n', encoding="utf-8")
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)

        def run(*args, cwd=None, check=True):
            result = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd,
                                    env=env, capture_output=True, text=True)
            if token:
                result.stderr = result.stderr.replace(token, "[REDACTED]")
                result.stdout = result.stdout.replace(token, "[REDACTED]")
            if check and result.returncode:
                detail = result.stderr.strip()
                raise RuntimeError("Git repository access failed. Check Kaggle Internet and GITHUB_TOKEN "
                                   "repository access (Contents: Read-only). " + detail)
            return result

        try:
            yield run
        finally:
            env.pop("GECC_GITHUB_TOKEN", None)
            token = None

if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")

with github_git() as git:
    remote = git("ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                 f"refs/heads/{BRANCH}", check=False)
    if remote.returncode == 2:
        raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
    if remote.returncode != 0:
        raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and grant GITHUB_TOKEN "
                           "access to this private repository with Contents: Read-only permission. "
                           f"Git exit code {remote.returncode}: {remote.stderr.strip()}")
    source_dir = Path(tempfile.mkdtemp(prefix="gecc-two-phase-source-", dir=SOURCE_ROOT))
    git("clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
        "--", REPO_URL, str(source_dir))
PROJECT_DIR = (source_dir / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(source_dir.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements-kaggle.txt", "model/classification.py", "model/correction.py",
                 "model/two_phase.py", "configs/baseline.json", "configs/correction.json", "configs/experiment.json", "tools/run_experiment.py"]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source_dir, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the notebook kernel untouched: Save Version cannot pause for a restart.
# Reuse Kaggle's CUDA torch through system-site-packages; install training pins locally.
from importlib.metadata import version
import tempfile

installed_torch = version("torch")
TRAINING_ENV = Path(tempfile.mkdtemp(prefix="gecc-training-env-"))
# Kaggle's system Python cannot reliably bootstrap pip through ensurepip.
# Create only the interpreter; the existing host pip manages this environment.
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages",
                str(TRAINING_ENV)], check=True)
TRAINING_PYTHON = str(TRAINING_ENV / "bin/python")
constraints = TRAINING_ENV / "constraints.txt"
constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "--python", TRAINING_PYTHON, "install",
                "-r", str(PROJECT_DIR / "requirements-kaggle.txt"),
                "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
subprocess.run([TRAINING_PYTHON, "-c",
                "import numpy, scipy, sklearn, torch, transformers, peft, accelerate, bitsandbytes, lmformatenforcer, wandb; "
                "from transformers import AutoModelForCausalLM; from sklearn.metrics import roc_curve; "
                f"assert torch.__version__ == {installed_torch!r}; "
                "print('Training imports passed; torch', torch.__version__, 'CUDA', torch.version.cuda, 'W&B', wandb.__version__)"], check=True)
print("Isolated training environment ready; no kernel restart needed.")



In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
launcher.kaggle_credentials()
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
